# 04 - Advanced Search Strategies

If you want to estimate a large number of models, it is recommended to run Delphos in a loop. Delphos caches Apollo estimation results in a local SQLite database, meaning that if it proposes the same model twice, it will not waste time estimating it again in R.

In this notebook, we'll see how to:
- Configure advanced parameters.
- Run Delphos in a loop.
- Save results to CSV incrementally.


In [ ]:
import delphos as dp
import pandas as pd

dataset = dp.load_dataset("dataset_4")
agent = dp.load_agent()


### 1. Advanced Parameters

Delphos exposes advanced sampling controls via `**advanced_kwargs` in the `propose()` method. 

- `epsilon`: (default `0.0`) Controls exploration vs exploitation. E.g. `epsilon=0.1` means 10% of the time, the agent takes a random action.
- `temperature`: (default `1.0`) Softmax temperature for sampling. Higher = more uniform, lower = more greedy.


In [ ]:
models = agent.propose(
    dataset,
    n_models=2,
    estimate=True,
    epsilon=0.05,       # 5% random exploration
    temperature=1.5     # slightly softer sampling
)
models.to_dataframe()


### 2. Strategy Schedules

Instead of using the same search parameters over and over, you can define a schedule to vary them. A common pattern is to start greedy, move to top-k sampling for diverse candidates, and finish with stochastic sampling to force exploration.

In [ ]:
strategy_schedule = [
    {"strategy": "greedy", "n_models": 1, "max_attempts": 10},
    {"strategy": "topk", "n_models": 2, "max_attempts": 100, "top_k": 5, "temperature": 0.8},
    {"strategy": "stochastic", "n_models": 2, "max_attempts": 100, "epsilon": 0.15},
]
strategy_schedule

### 3. Running a batch loop

To search extensively, place the proposal call inside a loop and append the results to a CSV. We rotate through the strategy schedule at each iteration.

In [ ]:
import os

output_csv = "advanced_search_results.csv"

n_iterations = 3
batch_size = 2

for i in range(n_iterations):
    settings = strategy_schedule[i % len(strategy_schedule)]
    print(f"Iteration {i+1}/{n_iterations} using settings: {settings}")
    
    batch = agent.propose(
        dataset,
        estimate=True,
        **settings
    )
    
    df = batch.to_dataframe()
    
    if not os.path.exists(output_csv):
        df.to_csv(output_csv, index=False)
    else:
        df.to_csv(output_csv, mode='a', header=False, index=False)

print("Search completed!")

### 4. The Result Cache

Why is it safe to loop like this overnight? Because Delphos maintains an internal SQLite database caching all estimation results! If it proposes a specification it has already seen, it immediately loads the results instead of calling R, saving an enormous amount of time.

In [ ]:
from delphos.env.result_cache import ResultCache

cache = ResultCache(dataset.rewards_path)
print("Cache path:", cache.db_path)
print("Rows currently stored:", len(cache.load(dataset.name)))


### 5. Reviewing unique results

Once your search finishes, load the CSV and drop duplicates to find the best models!

In [ ]:
results = pd.read_csv(output_csv)

unique_results = results.drop_duplicates(subset=["specification_key"])
best_models = unique_results.sort_values("BIC", ascending=True)

print(f"Found {len(unique_results)} unique specifications.")
best_models[["specification_key", "BIC", "AIC", "LLout"]].head(5)
